# 🧪 RAG Lab — Ollama & LangChain
**Build a Retrieval-Augmented Generation (RAG) App**

Using **Ollama** (local LLMs) + **LangChain** (RAG pipelines)

Domain: **Climate Science** (IPCC AR6 PDFs as knowledge base)

---

## Overview
Ce lab vous guide dans la construction d'une application RAG qui :
- ingère des PDFs externes (documents IPCC AR6),
- extrait et découpe le texte en chunks,
- crée des embeddings via un modèle Ollama local,
- stocke les embeddings dans un vector DB local (Chroma ou FAISS),
- expose une API RAG (Retriever + LLM) via FastAPI,
- fournit une interface minimale (Streamlit) pour les requêtes interactives.

## 1. Objectifs pédagogiques
À la fin de ce lab, vous serez capable de :
1. Installer et exécuter Ollama localement et télécharger des modèles adaptés à votre machine.
2. Utiliser LangChain (avec l'intégration Ollama) pour produire des embeddings et exécuter des modèles de chat.
3. Ingérer de longs PDFs, les découper en chunks et les persister dans un vector store.
4. Implémenter un pipeline RAG (retriever + LLM) et l'exposer via un endpoint REST.
5. Construire une interface minimale (Streamlit) qui interroge le backend et affiche les réponses et sources.

## 2. Prérequis
Vérifiez que vous avez :
- [ ] Python 3.10+ installé
- [ ] `git`, `curl` (ou `wget`), `node/npm` (optionnel, pour React)
- [ ] Assez d'espace disque (modèles + index) — préférez des modèles 1–7B
- [ ] Connaissance basique de Python, des environnements virtuels et des APIs REST

## 3. Dataset — IPCC AR6 (liens directs)
Téléchargez les PDFs officiels de l'IPCC et placez-les dans le dossier `data/` du projet.

| Document | Lien |
|---|---|
| **WGI SPM** — Physical Science Basis | https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_SPM.pdf |
| **AR6 Synthesis Report** — Full Volume | https://www.ipcc.ch/report/ar6/syr/downloads/report/IPCC_AR6_SYR_FullVolume.pdf |
| **AR6 SYR SPM** — Summary for Policymakers | https://www.ipcc.ch/report/ar6/syr/downloads/report/IPCC_AR6_SYR_SPM.pdf |

## 4. Architecture

```
PDFs  →  Embeddings  →  Vector DB  →  RAG chain  →  API  →  Frontend
```

| Composant | Rôle |
|---|---|
| **Ingest pipeline** | PDFs → extraction texte → chunking |
| **Embeddings** | Ollama embeddings (local) → vecteurs |
| **Vector DB** | Chroma ou FAISS pour la recherche par similarité |
| **RAG chain** | Retriever + LLM (ChatOllama) via LangChain |
| **API** | FastAPI enveloppe le pipeline |
| **Frontend** | Streamlit (React en option) |

## 5. Structure du projet

```
rag-ollama-ipcc/
├── data/            # PDFs IPCC ici
├── chunks/          # fichiers JSON intermédiaires
├── vectordb/        # vector DB persisté (Chroma)
├── ingest.py        # extraction & chunking des PDFs
├── embeddings.py    # embeddings & persistance vectordb
├── app.py           # backend FastAPI (endpoints ask/search)
├── ui_streamlit.py  # interface Streamlit
├── requirements.txt
└── README.md
```

---
## 6. Lab pas à pas
### A. Installer et démarrer Ollama

In [1]:
# Installer Ollama (Linux/macOS — exécutez dans un terminal, pas ici)
# curl -fsSL https://ollama.com/install.sh | sh

# Démarrer le daemon Ollama
# ollama serve &

# Vérifier l'installation
import subprocess
result = subprocess.run(["ollama", "-v"], capture_output=True, text=True)
print(result.stdout or result.stderr)

ollama version is 0.21.2



### B. Créer le projet et l'environnement virtuel

> Exécutez ces commandes dans un terminal :
```bash
mkdir rag-ollama-ipcc && cd rag-ollama-ipcc
python -m venv .venv
source .venv/bin/activate   # Windows : .venv\Scripts\activate
pip install -U pip
```

### C. Installer les dépendances Python

In [2]:
# Installer toutes les dépendances nécessaires
!pip install langchain chromadb unstructured fastapi "uvicorn[standard]" python-multipart PyPDF2 tiktoken streamlit langchain-ollama


[notice] A new release of pip available: 22.2.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### D. Télécharger les PDFs IPCC

In [3]:
import os

os.makedirs("data", exist_ok=True)

pdfs = {
    "data/WGI_SPM.pdf": "https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_SPM.pdf",
    "data/AR6_SYR_FullVolume.pdf": "https://www.ipcc.ch/report/ar6/syr/downloads/report/IPCC_AR6_SYR_FullVolume.pdf",
    "data/AR6_SYR_SPM.pdf": "https://www.ipcc.ch/report/ar6/syr/downloads/report/IPCC_AR6_SYR_SPM.pdf",
}

for path, url in pdfs.items():
    if not os.path.exists(path):
        print(f"Téléchargement de {path}...")
        os.system(f'wget -q -O "{path}" "{url}"')
        print(f"  ✅ Sauvegardé : {path}")
    else:
        print(f"  ℹ️  Déjà présent : {path}")

print("\nFichiers dans data/ :", os.listdir("data"))

  ℹ️  Déjà présent : data/WGI_SPM.pdf
  ℹ️  Déjà présent : data/AR6_SYR_FullVolume.pdf
  ℹ️  Déjà présent : data/AR6_SYR_SPM.pdf

Fichiers dans data/ : ['AR6_SYR_FullVolume.pdf', 'AR6_SYR_SPM.pdf', 'WGI_SPM.pdf']


### E. Ingestion — Extraction et chunking des PDFs

On utilise `UnstructuredPDFLoader` et `RecursiveCharacterTextSplitter` de LangChain.

In [9]:
from langchain_community.document_loaders import PyPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.document_loaders import PyPDFLoader


import os, glob, json


def load_and_split(pdf_path, chunk_size=1000, chunk_overlap=200):
    """Charge un PDF et le découpe en chunks."""
    #loader = UnstructuredPDFLoader(pdf_path)
    loader = PyPDFLoader("data/AR6_SYR_FullVolume.pdf")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_docs = splitter.split_documents(docs)
    return split_docs


def ingest_all(data_dir="data", chunks_dir="chunks", chunk_size=1000, chunk_overlap=200):
    """Traite tous les PDFs du dossier data/ et sauvegarde les chunks en JSON."""
    os.makedirs(chunks_dir, exist_ok=True)
    for p in glob.glob(f"{data_dir}/*.pdf"):
        print(f"Traitement de {p}...")
        docs = load_and_split(p, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        out = [{"page_content": d.page_content, "metadata": d.metadata} for d in docs]
        fn = os.path.join(chunks_dir, os.path.basename(p) + ".json")
        with open(fn, "w", encoding="utf8") as f:
            json.dump(out, f)
        print(f"  ✅ {len(docs)} chunks sauvegardés dans {fn}")


# Lancer l'ingestion
ingest_all()
print("\nIngestion terminée !")

Traitement de data\AR6_SYR_FullVolume.pdf...
  ✅ 879 chunks sauvegardés dans chunks\AR6_SYR_FullVolume.pdf.json
Traitement de data\AR6_SYR_SPM.pdf...
  ✅ 879 chunks sauvegardés dans chunks\AR6_SYR_SPM.pdf.json
Traitement de data\WGI_SPM.pdf...
  ✅ 879 chunks sauvegardés dans chunks\WGI_SPM.pdf.json

Ingestion terminée !


In [10]:
# Vérification : afficher un aperçu du premier chunk
import json, os

chunk_files = [f for f in os.listdir("chunks") if f.endswith(".json")]
if chunk_files:
    with open(os.path.join("chunks", chunk_files[0]), "r", encoding="utf8") as f:
        sample = json.load(f)
    print(f"Fichier : {chunk_files[0]}")
    print(f"Nombre de chunks : {len(sample)}")
    print(f"\n--- Aperçu du premier chunk ---\n")
    print(sample[0]["page_content"][:500])
else:
    print("Aucun fichier de chunks trouvé.")

Fichier : AR6_SYR_FullVolume.pdf.json
Nombre de chunks : 879

--- Aperçu du premier chunk ---

1
CLIMATE CHANGE 2023
Synthesis Report
A Report of the Intergovernmental Panel on Climate Change


### F. Créer les embeddings et persister dans le Vector DB

> **Note instructor** : vérifiez le nom du modèle d'embedding disponible via `ollama ls` et adaptez le script.

On utilise le modèle `nomic-embed-text` et Chroma comme vector store.

In [11]:
# Télécharger le modèle d'embedding (à exécuter une seule fois)
!ollama pull nomic-embed-text:latest

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 970aa74c0a90:   0% ▕                  ▏ 297 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 1.5 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 2.2 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   1% ▕                  ▏ 3.4 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   2% ▕                  ▏ 4.7 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   2% ▕                  ▏ 5.2 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   2% ▕                  ▏ 5.9 MB/274

In [14]:
# embeddings.py — contenu complet

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import json, os


def embed_and_store(
    chunks_dir="chunks",
    persist_directory="vectordb",
    embedding_model="nomic-embed-text:latest"
):
    """Produit les embeddings de tous les chunks et les persiste dans Chroma."""
    embedder = OllamaEmbeddings(model=embedding_model)
    documents = []

    for fn in os.listdir(chunks_dir):
        if not fn.endswith(".json"):
            continue
        with open(os.path.join(chunks_dir, fn), "r", encoding="utf8") as f:
            items = json.load(f)
        for it in items:
            documents.append(
                Document(
                    page_content=it["page_content"],
                    metadata=it.get("metadata", {})
                )
            )

    print(f"Nombre total de documents à encoder : {len(documents)}")
    vectordb = Chroma.from_documents(
        documents,
        embedding=embedder,
        persist_directory=persist_directory
    )
    vectordb.persist()
    print(f"✅ Vector DB persisté dans '{persist_directory}'")
    return vectordb


# Lancer la création des embeddings
vectordb = embed_and_store()

Nombre total de documents à encoder : 2637
✅ Vector DB persisté dans 'vectordb'


C:\Users\MoustaphaCOMPAORE\AppData\Local\Temp\ipykernel_20328\56041750.py:37: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [15]:
# Test rapide : recherche par similarité
test_query = "What are the main causes of climate change?"
results = vectordb.similarity_search(test_query, k=2)

print(f"Résultats pour : '{test_query}'\n")
for i, doc in enumerate(results):
    print(f"--- Résultat {i+1} ---")
    print(doc.page_content[:300])
    print(f"Métadonnées : {doc.metadata}\n")

Résultats pour : 'What are the main causes of climate change?'

--- Résultat 1 ---
virtually certain that human-caused CO 2 emissions are the main driver 
of current global acidification of the surface open ocean. {WGI SPM A.1, 
WGI SPM A.1.3, WGI SPM A.1.5, WGI SPM A.1.6, WG1 SPM A1.7, 
WGI SPM A.2, WG1.SPM A.4.2; SROCC SPM.A.1, SROCC SPM A.2}
Human-caused climate change is alrea
Métadonnées : {'creator': 'Adobe InDesign 18.3 (Macintosh)', 'page': 61, 'creationdate': '2023-07-15T11:35:23+09:00', 'producer': 'iLovePDF', 'trapped': '/False', 'page_label': '46', 'source': 'data/AR6_SYR_FullVolume.pdf', 'total_pages': 186, 'moddate': '2023-08-15T19:45:58+09:00'}

--- Résultat 2 ---
virtually certain that human-caused CO 2 emissions are the main driver 
of current global acidification of the surface open ocean. {WGI SPM A.1, 
WGI SPM A.1.3, WGI SPM A.1.5, WGI SPM A.1.6, WG1 SPM A1.7, 
WGI SPM A.2, WG1.SPM A.4.2; SROCC SPM.A.1, SROCC SPM A.2}
Human-caused climate change is alrea
Métadonnées

### G. Backend FastAPI — Endpoints RAG

Le code ci-dessous définit `app.py`. Sauvegardez-le dans un fichier séparé et lancez-le via `uvicorn`.

In [ ]:
# Écrire app.py sur disque
app_code = '''
# app.py
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

app = FastAPI(title="RAG IPCC API")

# Charger le vector DB et configurer le retriever
embedding_fn = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma(persist_directory="vectordb", embedding_function=embedding_fn)
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# LLM
llm = ChatOllama(model="llama3.1", temperature=0.0)

prompt = PromptTemplate.from_template(
    "Use the following context to answer the question. "
    "If the answer is not in the context, say \'I don\'t know.\'\\n\\n"
    "Context:\\n{context}\\n\\nQuestion: {question}"
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)


class QueryIn(BaseModel):
    question: str


@app.get("/")
def root():
    return {"message": "RAG IPCC API is running. POST /ask to query."}


@app.post("/ask")
def ask(q: QueryIn):
    result = qa({"query": q.question})
    return {
        "answer": result["result"],
        "sources": [doc.metadata for doc in result.get("source_documents", [])]
    }
'''

with open("app.py", "w", encoding="utf8") as f:
    f.write(app_code)

print("✅ app.py créé.")
print("Lancez le backend dans un terminal avec :")
print("  uvicorn app:app --reload --port 8000")

In [ ]:
# (Optionnel) Lancer le serveur FastAPI en arrière-plan depuis le notebook
import subprocess, time

server = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(3)  # attendre que le serveur démarre
print("✅ Serveur FastAPI démarré sur http://localhost:8000")

In [ ]:
# Tester l'API via requests
import requests

question = "What does the IPCC say about global warming of 1.5°C?"

response = requests.post(
    "http://localhost:8000/ask",
    json={"question": question}
)

if response.ok:
    data = response.json()
    print("Question :", question)
    print("\nRéponse :\n", data["answer"])
    print("\nSources :")
    for s in data["sources"]:
        print(" -", s)
else:
    print("Erreur API :", response.status_code, response.text)

### H. Interface Streamlit

Créez `ui_streamlit.py` avec le code ci-dessous, puis lancez : `streamlit run ui_streamlit.py`

In [ ]:
# Écrire ui_streamlit.py sur disque
streamlit_code = '''
# ui_streamlit.py
import streamlit as st
import requests

st.set_page_config(page_title="RAG IPCC", page_icon="🌍")
st.title("🌍 RAG Demo — IPCC AR6")
st.caption("Powered by Ollama + LangChain")

q = st.text_input("Posez une question sur les rapports IPCC")

if st.button("Demander") and q:
    with st.spinner("Recherche en cours..."):
        try:
            resp = requests.post(
                "http://localhost:8000/ask",
                json={"question": q},
                timeout=60
            )
            if resp.ok:
                data = resp.json()
                st.subheader("Réponse")
                st.write(data["answer"])
                if data.get("sources"):
                    st.subheader("Sources")
                    for src in data["sources"]:
                        st.json(src)
            else:
                st.error(f"Erreur API : {resp.status_code}")
        except requests.exceptions.ConnectionError:
            st.error("Impossible de contacter le backend. Assurez-vous qu\'uvicorn tourne sur le port 8000.")
'''

with open("ui_streamlit.py", "w", encoding="utf8") as f:
    f.write(streamlit_code)

print("✅ ui_streamlit.py créé.")
print("Lancez l'interface avec :")
print("  streamlit run ui_streamlit.py")

---
## 7. Notes & Dépannage

| Problème | Solution |
|---|---|
| **Modèle trop lourd** | Utilisez des modèles 1–7B (ex: `llama3.2:1b`, `phi3`) |
| **Erreurs `unstructured`** | Utilisez Docker ou le fournisseur conda |
| **Nom du modèle d'embedding inconnu** | Vérifiez avec `ollama ls` et adaptez le script |
| **Chroma vs FAISS** | Chroma est plus simple ; FAISS est plus rapide mais nécessite plus de compilation |
| **Taille des chunks** | Démarrez avec 800–1200 chars, overlap 150–300 chars |

---
## 8. Exercices supplémentaires

1. **Citations dans les réponses** : retournez `source_documents` avec numéros de page et affichez-les dans l'UI.
2. **Re-ranking** : implémentez un re-ranker avec un cross-encoder et comparez la qualité.
3. **Amélioration des prompts** : concevez des templates qui produisent des réponses concises et bien citées.
4. **Boucle de feedback utilisateur** : ajoutez un bouton "utile / pas utile" et loguez les réponses.
5. **React front end** : construisez une UI moderne (Vite + React) avec liens cliquables vers les sections PDF.
6. **Comparaison de frameworks** : reconstruisez le même pipeline avec **LlamaIndex** et comparez l'ergonomie.

---
## 9. Livrables

Chaque étudiant ou groupe doit soumettre :
1. Un **dépôt GitHub** contenant : `ingest.py`, `embeddings.py`, `app.py`, `ui_streamlit.py`, `requirements.txt` et `README.md` avec les étapes d'exécution.
2. Un **rapport** expliquant les choix de conception (taille des chunks, modèle d'embedding, paramètres du retriever) et 3 exemples de requêtes + sorties du modèle avec évaluation.

---
## 10. Commandes utiles (référence rapide)

```bash
# Démarrer le daemon Ollama
ollama serve &

# Lister les modèles locaux
ollama ls

# Télécharger un modèle
ollama pull <model-name>

# Lancer le backend FastAPI
uvicorn app:app --reload --port 8000

# Lancer l'interface Streamlit
streamlit run ui_streamlit.py
```

---
## 11. Appendice — Template README

```markdown
# RAG Ollama Lab — README

## Setup
1. Installer Ollama et démarrer le daemon : `ollama serve`
2. Créer et activer l'environnement virtuel : `python -m venv .venv && source .venv/bin/activate`
3. Installer les dépendances : `pip install -r requirements.txt`
4. Placer les PDFs IPCC dans `data/`

## Exécution
1. `python ingest.py`
2. `python embeddings.py`
3. `uvicorn app:app --reload --port 8000`
4. `streamlit run ui_streamlit.py`
```